# Phase 2 — Serving baseline + single-optimization marginals

Runs EXPERIMENT_MATRIX.md blocks 1–2 (IMPLEMENTATION_PLAN.md Phase 2) on the anchor model
**Llama-3.1-8B-Instruct**: the FP16/no-opt serving baseline plus each optimization ALONE —
W (AWQ W4A16), K (FP8 KV-cache, software-emulated on A100 — a documented deployment
condition, EXPERIMENT_MATRIX §7), S (EAGLE-3) — across GSM8K, HumanEval, and the RAG
shared-prefix workload at concurrency {1, 8, 32, 64}.

**48 run cells, 4 server launches.** Every run records goodput (verified-and-generated
tokens/sec) and the **measured emergent batch size** — concurrency is set, batch size is
never set (PROJECT_SPEC §7.2).

**Requirements**
- **A100 runtime** — verify with the cell below, don't trust the dropdown.
- **HF token** as Colab secret `HF_TOKEN`, with the **Llama-3.1** license accepted
  (separate gate from Llama-3: visit meta-llama/Llama-3.1-8B-Instruct and accept).

**Budget:** ~2.5–3.5 A100-hours ≈ 30–45 units at the calibrated ~12 units/hr
(PREREQ_RESULTS Check 1). Fully resumable: a disconnect costs one cell — re-run the
sweep cell and completed cells are skipped. Note the units meter before/after.

In [1]:
# 1. Verify the GPU actually assigned
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
units_before = input("Compute-units balance right now: ")

NVIDIA A100-SXM4-80GB, 81920 MiB
Compute-units balance right now: 165.61


In [2]:
# 2. Get the repo + Spec-Bench (RAG passages come from its 'rag' subtask;
#    external/ is git-ignored so it must be cloned per-session)
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench 2>/dev/null || echo "Spec-Bench already present"
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK"

/content
/content/repo
RAG data OK


In [3]:
# Cell 2 fix:
# Clean up the wrongly-placed files from the failed attempt
!rm -rf /content/repo /content/external

%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo
%cd /content/repo
!pwd && ls

/content
Cloning into 'repo'...
remote: Enumerating objects: 484, done.
remote: Counting objects: 100% (484/484), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 484 (delta 400), reused 459 (delta 379), pack-reused 0 (from 0)
Receiving objects: 100% (484/484), 230.06 KiB | 11.50 MiB/s, done.
Resolving deltas: 100% (400/400), done.
/content/repo
/content/repo
analysis	      HARNESS_SPEC.md	      requirements-dev.txt
block0_results	      IMPLEMENTATION_PLAN.md  scripts
colab		      LITERATURE.md	      SpeculativeDec_Testing.ipynb
configs		      PREREQ_RESULTS.md       tests
EXPERIMENT_MATRIX.md  PROJECT_SPEC.md
harness		      README.md


In [4]:
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK"

Cloning into 'external/Spec-Bench'...
remote: Enumerating objects: 174, done.
remote: Counting objects: 100% (174/174), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 174 (delta 56), reused 147 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (174/174), 3.31 MiB | 34.25 MiB/s, done.
Resolving deltas: 100% (56/56), done.
RAG data OK


In [5]:
# 3. Isolated vLLM environment (PREREQ_RESULTS Check 6 recipe; ~6-8 min)
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 48.7 MB/s eta 0:00:00
vllm 0.24.0


In [6]:
# 4. HF auth (gated Llama-3.1 weights)
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

token set: True


In [7]:
# 5. Harness self-check (no GPU needed, ~1 min). Includes the RAG
#    byte-identical-prefix unit tests. Fix anything here before burning GPU time.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

........................................................................ [ 65%]
......................................                                   [100%]
110 passed in 9.64s


In [8]:
# 6. Shared-prefix token-ID check with the REAL Llama-3.1 tokenizer
#    (HARNESS_SPEC §7/§10 guard, beyond the synthetic-tokenizer unit test)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python scripts/check_rag_prefix.py





OK: prefix token-ID equality verified across 2 document groups with tokenizer meta-llama/Llama-3.1-8B-Instruct


In [9]:
# 7. Sanity: the four server commands (nothing launches)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_base_*_r0.yaml" "configs/factorial/cube_w_*_r0.yaml" "configs/factorial/cube_k_*_r0.yaml" "configs/factorial/cube_s_*_r0.yaml" --results-dir results --dry-run 2>&1 | grep "server command"

[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching
[sweep] server command: vllm serve hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --quantization awq_marlin
[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --kv-cache-dtype fp8
[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle3", "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B", 

In [10]:
# 8. THE SWEEP: 48 cells, 4 server groups, resumable. ~2.5-3.5 h.
# Order: baseline group first, then K (fp8 kv), then S (eagle3), then W (AWQ).
# If Colab disconnects: re-run this cell; completed cells are skipped.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_base_*_r0.yaml" "configs/factorial/cube_w_*_r0.yaml" "configs/factorial/cube_k_*_r0.yaml" "configs/factorial/cube_s_*_r0.yaml" --results-dir results

[sweep] 48 run(s) in 4 server group(s)
[sweep] group 1/4: 12 pending run(s)
[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching
[sweep] waiting for server (log: results/server_logs/server_20260708_214450.log)





[run] core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c1_r0: 64 prompts, concurrency=1, max_new_tokens=256
  [load] 25/64 requests done
  [load] 50/64 requests done
[run] core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c1_r0 -> results/runs/core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c1_r0.json (status=ok, 90.5 tok/s mean/request, tau=None, acc=0.797)
[run] core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c32_r0: 320 prompts, concurrency=32, max_new_tokens=256
  [load] 25/320 requests done
  [load] 50/320 requests done
  [load] 75/320 requests done
  [load] 100/320 requests done
  [loa

In [11]:
# 9. Marginals report: goodput vs concurrency per optimization, with
#    speedup vs baseline, emergent batch size, and tau for the EAGLE-3 cells
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.marginals results

# Phase-2 marginals: goodput vs concurrency

## gsm8k

| conc | baseline goodput (xbase) | W (w4a16) goodput (xbase) | K (fp8-kv) goodput (xbase) | S (eagle3) goodput (xbase) | emergent batch (mean/max, baseline) | tau (S) |
|---|---|---|---|---|---|---|
| 1 | 90 | 187 (2.07x) | 85 (0.94x) | 193 (2.13x) | 1.0 / 1 | 2.85 |
| 8 | 626 | 1065 (1.70x) | 593 (0.95x) | 1099 (1.76x) | 7.6 / 8 | 2.88 |
| 32 | 1591 | 2017 (1.27x) | 1566 (0.98x) | 1944 (1.22x) | 30.2 / 32 | 2.89 |
| 64 | 2237 | 2259 (1.01x) | 2291 (1.02x) | 2173 (0.97x) | 58.9 / 64 | 2.89 |

## humaneval

| conc | baseline goodput (xbase) | W (w4a16) goodput (xbase) | K (fp8-kv) goodput (xbase) | S (eagle3) goodput (xbase) | emergent batch (mean/max, baseline) | tau (S) |
|---|---|---|---|---|---|---|
| 1 | 92 | 196 (2.13x) | 86 (0.94x) | 290 (3.16x) | 1.0 / 1 | 4.09 |
| 8 | 688 | 1340 (1.95x) | 651 (0.95x) | 1874 (2.72x) | 7.7 / 8 | 4.08 |
| 32 | 2127 | 3331 (1.57x) | 2002 (0.94x) | 3924 (1.84x) | 29.6 / 32 | 4.08 |
| 64 | 3468 

In [12]:
# 10. Preserve everything
units_after = input("Compute-units balance now: ")
print("Units burned:", units_before, "->", units_after)
!zip -qr phase2_results.zip results
from google.colab import files
files.download("phase2_results.zip")

Compute-units balance now: 153.88
Units burned: 165.61 -> 153.88


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reading the result

The marginals table is the Phase-2 deliverable. What to look for:

- **S (EAGLE-3) column:** relative speedup should be largest at conc=1 and erode as
  concurrency grows (the serving-regime thesis). Find the crossover — the concurrency
  where it drops below ~1.0x — per workload; that number is hypothesis "S main"
  (EXPERIMENT_MATRIX §5) answered.
- **W (AWQ) column:** strong win at conc=1 (memory-bound decode); watch whether it
  shrinks or flips at conc 32–64 as the regime turns compute-bound — dequant overhead
  is the same mechanism as the Block-0 finding, now on the serving axis.
- **K (FP8-KV) column:** expect ≈neutral speed on short prompts (GSM8K/HumanEval) —
  it's emulated on A100 — but a real gain on RAG at high concurrency via *capacity*:
  compare the **emergent batch** columns; FP8-KV should sustain a larger running batch
  before preemption.
- **Emergent batch vs concurrency 64 on RAG:** expect batch ≪ 64 (KV capacity limits
  it) — that gap is the capacity mechanism made visible, and it's the reason batch
  size must be measured, not set.
- **tau across concurrency:** should be roughly stable — if it collapses at high
  concurrency, inspect before interpreting (scheduler behavior, not acceptance).

Afterwards: commit `results/` runs + the report (or the zip), update PREREQ_RESULTS
Check 1 with this session's burn rate, and Phase 3 is just the remaining four corners
of the cube through the same sweep machinery + `analysis/factorial.py`.